<a href="https://colab.research.google.com/github/simulate111/Computer-vision2026ABO/blob/main/CompViz_part_I_nr_5_ipynb_txt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IT00CJ11 Computer Vision (5 cr. ECTS), spring 2025

***Direct computational methods and deep learning for vision***

***Image to image mappings***

We will now take a look at transformations between images and a few practical methods for computing them.

*Homographies*

A homography is a 2D projective transformation that maps points in one plane to another. In our case, the planes are images or planar surfaces in 3D. Homographies have many practical uses, such as registering images, rectifying images, texture warping, and creating panoramas.We will make frequent use of them. In essence, a homography $H$ maps 2D points (in homogeneous coordinates) according to:

$$\left( \begin{array}{c} x' \\ y' \\ w' \end{array} \right) =
\left( \begin{array}{ccc} h_1 & h_2 & h_3 \\ h_4 & h_5 & h_6 \\ h_7 & h_8 & h_9 \end{array} \right)
\left( \begin{array}{c} x \\ y \\ w \end{array} \right) \ \text{or} \ \mathbf{x}' = H \mathbf{x} \ .$$

Homogeneous coordinates are a useful representation for points in image planes (and in 3D, as we will see later). Points in homogeneous coordinates are only defined up to scale so that

$$\mathbf{x} = \left( \begin{array}{ccc} x & y & w \end{array} \right) = \left( \begin{array}{ccc} \alpha x & \alpha y & \alpha w \end{array} \right) = \left( \begin{array}{ccc} \frac{x}{w} & \frac{y}{w} & 1 \end{array} \right)$$

all refer to the same 2D point. As a consequence, the homography $H$ is also only defined up to scale and has eight independent degrees of freedom. Often points are normalized with $w = 1$ to have a unique identification of the image coordinates $x$, $y$. The extra coordinate makes it easy to represent transformations with a single matrix.

The following scripts normalize and convert points to homogeneous coordinates:

In [1]:
def normalize(points):
    """ Normalize a collection of points in
        homogeneous coordinates so that last row = 1. """

    for row in points:
        row /= points[-1]
    return points


def make_homog(points):
    """ Convert a set of points (dim*n array) to
        homogeneous coordinates. """

    return vstack((points,ones((1,points.shape[1]))))

When working with points and transformations, we will store the points column-wise so that a set of $n$ points in two dimensions will be a $3 \times n$ array in homogeneous coordinates. This format makes matrix multiplications and point transforms easier. For all other cases, we will typically use rows to store data, for example features for clustering and classification.

There are some important special cases of these projective transformations. An *affine
transformation*:

$$\left( \begin{array}{c} x' \\ y' \\ 1 \end{array} \right) =
\left( \begin{array}{ccc} a_1 & a_2 & t_x \\ a_3 & a_4 & t_y \\ 0 & 0 & 1 \end{array} \right)
\left( \begin{array}{c} x \\ y \\ 1 \end{array} \right) \ \text{or} \
\mathbf{x}' = \left( \begin{array}{cc} \mathbf{A} & \mathbf{t} \\ \mathbf{0} & 1 \end{array} \right) \mathbf{x} \ ,$$

preserves $w = 1$ and cannot represent as strong deformations as a full projective transformation. The affine transformation contains an invertible matrix $\mathbf{A}$ and a translation vector $\mathbf{t} = \left( \begin{array}{cc} t_x & t_y \end{array} \right)$. Affine transformations are used, for example, in warping.

A *similarity transformation*:

$$\left( \begin{array}{c} x' \\ y' \\ 1 \end{array} \right) =
\left( \begin{array}{ccc} s \cos{\theta} & -s \sin{\theta} & t_x \\ s \sin{\theta} & s \cos{\theta} & t_y \\ 0 & 0 & 1 \end{array} \right)
\left( \begin{array}{c} x \\ y \\ 1 \end{array} \right) \ \text{or} \
\mathbf{x}' = \left( \begin{array}{cc} s \mathbf{R} & \mathbf{t} \\ \mathbf{0} & 1 \end{array} \right) \mathbf{x} \ ,$$

is a rigid 2D transformation that also includes scale changes. The scalar $s$ specifies scaling, $\mathbf{R}$ is a rotation of an angle $\theta$, and $\mathbf{t} = \left( \begin{array}{cc} t_x & t_y \end{array} \right)$ is again a translation.With $s = 1$ distances are preserved and it is then a rigid transformation. Similarity transformations are used, e.g., in image registration.

Let’s look at algorithms for estimating homographies and then go into examples of using affine transformations for warping, similarity transformations for registration, and finally full projective transformations for creating panoramas.

*The Direct Linear Transformation Algorithm*

Homographies can be computed directly from corresponding points in two images (or planes). As mentioned earlier, a full projective transformation has eight degrees of freedom. Each point correspondence gives two equations, one each for the $x$- and $y$-coordinates, and therefore four point correspondences are needed to compute $H$.


The direct linear transformation (DLT) is an algorithm for computing $H$ given four or more correspondences. By rewriting the equation for mapping points using $H$ for several correspondences, we get an equation like

![image.png](attachment:image.png)

or $\mathbf{A} \mathbf{h} = 0$ where $\mathbf{A}$ is a matrix with twice as many rows as correspondences. By stacking
all corresponding points, a least squares solution for $H$ can be found using singular value decomposition (SVD). Here’s what it looks like in code.

In [2]:
def H_from_points(fp,tp):
    """ Find homography H, such that fp is mapped to tp
        using the linear DLT method. Points are conditioned
        automatically. """

    if fp.shape != tp.shape:
        raise RuntimeError('number of points do not match')

    # condition points (important for numerical reasons)
    # --from points--
    m = mean(fp[:2], axis=1)
    maxstd = max(std(fp[:2], axis=1)) + 1e-9
    C1 = diag([1/maxstd, 1/maxstd, 1])
    C1[0][2] = -m[0]/maxstd
    C1[1][2] = -m[1]/maxstd
    fp = dot(C1,fp)

    # --to points--
    m = mean(tp[:2], axis=1)
    maxstd = max(std(tp[:2], axis=1)) + 1e-9
    C2 = diag([1/maxstd, 1/maxstd, 1])
    C2[0][2] = -m[0]/maxstd
    C2[1][2] = -m[1]/maxstd
    tp = dot(C2,tp)

    # create matrix for linear method, 2 rows for each correspondence pair
    nbr_correspondences = fp.shape[1]
    A = zeros((2*nbr_correspondences,9))
    for i in range(nbr_correspondences):
        A[2*i] = [-fp[0][i],-fp[1][i],-1,0,0,0,
                    tp[0][i]*fp[0][i],tp[0][i]*fp[1][i],tp[0][i]]
        A[2*i+1] = [0,0,0,-fp[0][i],-fp[1][i],-1,
                    tp[1][i]*fp[0][i],tp[1][i]*fp[1][i],tp[1][i]]

    U,S,V = linalg.svd(A)
    H = V[8].reshape((3,3))

    # decondition
    H = dot(linalg.inv(C2),dot(H,C1))

    # normalize and return
    return H / H[2,2]

The first thing that happens in this function is a check that the number of points are equal. If not, an exception is thrown. This is useful for writing robust code, but we will only use exceptions in very few cases in this book to make the code samples simpler and easier to follow. You can read more about exception types at
http://docs.python.org/library/exceptions.html and how to use them at
http://docs.python.org/tutorial/errors.html.

The points are conditioned by normalizing so that they have zero mean and unit
standard deviation. This is very important for numerical reasons, since the stability of the algorithm is dependent on the coordinate representation. Then the matrix $\mathbf{A}$ is created using the point correspondences. The least squares solution is found as the last row of the matrix $\mathbf{V}$ of the SVD. The row is reshaped to create $H$. This matrix is then de-conditioned and normalized before being returned.


*Affine Transformations*

An affine transformation has six degrees of freedom and therefore three point correspondences are needed to estimate $H$. Affine transforms can be estimated using the DLT algorithm above by setting the last two elements equal to zero, $h_7 = h_8 = 0$.

Here we will use a different approach. Consider the following function, which computes the affine transformation matrix
from point correspondences:

In [3]:
def Haffine_from_points(fp,tp):
    """ Find H, affine transformation, such that
        tp is affine transf of fp. """

    if fp.shape != tp.shape:
        raise RuntimeError('number of points do not match')

    # condition points
    # --from points--
    m = mean(fp[:2], axis=1)
    maxstd = max(std(fp[:2], axis=1)) + 1e-9
    C1 = diag([1/maxstd, 1/maxstd, 1])
    C1[0][2] = -m[0]/maxstd
    C1[1][2] = -m[1]/maxstd
    fp_cond = dot(C1,fp)

    # --to points--
    m = mean(tp[:2], axis=1)
    C2 = C1.copy() #must use same scaling for both point sets
    C2[0][2] = -m[0]/maxstd
    C2[1][2] = -m[1]/maxstd
    tp_cond = dot(C2,tp)

    # conditioned points have mean zero, so translation is zero
    A = concatenate((fp_cond[:2],tp_cond[:2]), axis=0)
    U,S,V = linalg.svd(A.T)

    # create B and C matrices as Hartley-Zisserman (2:nd ed) p 130.
    tmp = V[:2].T
    B = tmp[:2]
    C = tmp[2:4]

    tmp2 = concatenate((dot(C,linalg.pinv(B)),zeros((2,1))), axis=1)
    H = vstack((tmp2,[0,0,1]))

    # decondition
    H = dot(linalg.inv(C2),dot(H,C1))

    return H / H[2,2]

Again, the points are conditioned and de-conditioned as in the DLT algorithm. Let’s see what these affine transformations can do with images in the next section.

*Warping Images*

Applying an affine transformation matrix $H$ on image patches is called warping (or affine warping) and is frequently used in computer graphics but also in several computer vision algorithms. A warp can easily be performedwith SciPy using the ndimage package.

The command

transformed_im = ndimage.affine_transform(im,A,b,size)

transforms the image patch im with A a linear transformation and b a translation vector as above. The optional argument size can be used to specify the size of the output image. The default is an image with the same size as the original. To see how this works, try running:

In [4]:
from PIL import Image
from pylab import *
from scipy import ndimage
im = array(Image.open('empire.jpg').convert('L'))
H = array([[1.4,0.05,-100],[0.05,1.5,-100],[0,0,1]])
im2 = ndimage.affine_transform(im,H[:2,:2],(H[0,2],H[1,2]))
figure()
gray()
imshow(im2)
show()

FileNotFoundError: [Errno 2] No such file or directory: 'empire.jpg'

As seen from the image, missing pixel values are set to zero.

*Image in Image*

A simple example of affine warping is to place images, or parts of images, inside another image so that they line up with specific areas or landmarks. The following function takes two images and the corner coordinates of where to put the first image in the second:

In [ ]:
def image_in_image(im1,im2,tp):
    """ Put im1 in im2 with an affine transformation
        such that corners are as close to tp as possible.
        tp are homogeneous and counter-clockwise from top left. """

    # points to warp from
    m,n = im1.shape[:2]
    fp = array([[0,m,m,0],[0,0,n,n],[1,1,1,1]])

    # compute affine transform and apply
    H = Haffine_from_points(tp,fp)
    im1_t = ndimage.affine_transform(im1,H[:2,:2],
                    (H[0,2],H[1,2]),im2.shape[:2])
    alpha = (im1_t > 0)

    return (1-alpha)*im2 + alpha*im1_t

As you can see, there is not much needed to do this. When blending together the warped image and the second image, we create an alpha map that defines how much of each pixel to take from each image. Here we use the fact that the warped image is filled with zeros outside the borders of the warped area to create a binary alpha map. To be really strict, we could have added a small number to the potential zero pixels of the first image, or done it properly (see exercises at the end of the chapter). Note that the image coordinates are in homogeneous form.

To try this function, let’s insert an image on a billboard in another image. The coordinates were determined manually by looking at a plot of the image (in PyLab figures, the mouse coordinates are shown near the bottom). PyLab’s ginput() could, of course, also have been used.

In [ ]:
from PIL import Image
from pylab import *
from numpy import *
from scipy import ndimage

# example of affine warp of im1 onto im2
im1 = array(Image.open('gubbe.jpg').convert('L'))
im2 = array(Image.open('billboard_for_rent.jpg').convert('L'))
# set to points
# tp = array([[264,538,540,264],[40,36,605,605],[1,1,1,1]])
# tp = array([[675,826,826,677],[55,52,281,277],[1,1,1,1]])
tp = array([[664,1440,1440,664],[140,156,805,805],[1,1,1,1]])
im3 = image_in_image(im1,im2,tp)
figure()
gray()
imshow(im3)
axis('equal')
axis('off')
show()

The function Haffine_from_points() gives the best affine transform for the given point correspondences. In the example above, those were the image corners and the corners of the billboard. If the perspective effects are small, this will give good results. The top row of Figure 3-3 in Solem shows what happens if we try to use an affine transformation to a billboard image with more perspective. It is not possible to transform all four corner points to their target locations with the same affine transform (a full projective transform would
have been able to do this, though). If you want to use an affine warp so that all corner points match, there is a useful trick.

For three points, an affine transform can warp an image so that the three correspondences match perfectly. This is because an affine transform has six degrees of freedom and three correspondences give exactly six constraints (x and y coordinates must match for all three). So if you really want the image to fit the billboard using affine transforms, you can divide the image into two triangles and warp them separately. Here’s how to do it:

In [ ]:
def alpha_for_triangle(points,m,n):
    """ Creates alpha map of size (m,n)
        for a triangle with corners defined by points
        (given in normalized homogeneous coordinates). """

    alpha = zeros((m,n))
    for i in range(min(points[0]),max(points[0])):
        for j in range(min(points[1]),max(points[1])):
            x = linalg.solve(points,[i,j,1])
            if min(x) > 0: #all coefficients positive
                alpha[i,j] = 1
    return alpha

# set from points to corners of im1
m,n = im1.shape[:2]
fp = array([[0,m,m,0],[0,0,n,n],[1,1,1,1]])
# first triangle
tp2 = tp[:,:3]
fp2 = fp[:,:3]
# compute H
H = Haffine_from_points(tp2,fp2)
im1_t = ndimage.affine_transform(im1,H[:2,:2],
(H[0,2],H[1,2]),im2.shape[:2])
# alpha for triangle
alpha = alpha_for_triangle(tp2,im2.shape[0],im2.shape[1])
im3 = (1-alpha)*im2 + alpha*im1_t
# second triangle
tp2 = tp[:,[0,2,3]]
fp2 = fp[:,[0,2,3]]

# compute H
H = Haffine_from_points(tp2,fp2)
im1_t = ndimage.affine_transform(im1,H[:2,:2],
(H[0,2],H[1,2]),im2.shape[:2])
# alpha for triangle
alpha = alpha_for_triangle(tp2,im2.shape[0],im2.shape[1])
im4 = (1-alpha)*im3 + alpha*im1_t
figure()
gray()
imshow(im4)
axis('equal')
axis('off')
show()

Here we simply create the alpha map for each triangle and then merge all images together. The alpha map for a triangle can be computed simply by checking if a pixel’s coordinates can bewritten as a convex combination of the triangle’s corner points. If the coordinates can be expressed this way, that means the pixel is inside the triangle. As seem from the figure, the corners should now match.

*Registering Images*

This is the process of transferring images so that they are aligned in a common coordinate frame. Registration can be rigid or non-rigid and is an important step in order to be able to do image comparisons and more sophisticated analysis.

Let’s look at an example of rigidly registering a set of face images so thatwe can compute the mean face and face appearance variations in a meaningful way. In this type of registration we are actually looking for a similarity transform (rigid with scale) to map correspondences. This is because the faces are not all at the same size, position, and rotation in the images.

In the file jkfaces.zip are 366 images of a single person (one for each day in 2008). The images are annotated with eye and mouth coordinates in the file jkfaces.xml. Using the points, a similarity transformation can be computed and the images warped to a normalized coordinate frame using this transformation (which, as mentioned, includes scaling). To read XML files, we will use minidom that comes with Python’s built-in xml.dom module.

The XML file looks like this:

![image.png](attachment:image.png)

To read the coordinates from the file, we use the following function that uses minidom:

In [ ]:
from PIL import Image
from xml.dom import minidom
from numpy import *
from pylab import *

from scipy import ndimage, linalg
import cv2
import os

def read_points_from_xml(xmlFileName):
    """ Reads control points for face alignment. """

    xmldoc = minidom.parse(xmlFileName)
    facelist = xmldoc.getElementsByTagName('face')
    faces = {}
    for xmlFace in facelist:
        fileName = xmlFace.attributes['file'].value
        xf = int(xmlFace.attributes['xf'].value)
        yf = int(xmlFace.attributes['yf'].value)
        xs = int(xmlFace.attributes['xs'].value)
        ys = int(xmlFace.attributes['ys'].value)
        xm = int(xmlFace.attributes['xm'].value)
        ym = int(xmlFace.attributes['ym'].value)
        faces[fileName] = array([xf, yf, xs, ys, xm, ym])
    return faces

The landmark points are returned in a Python dictionary with the filename of the image as key. The format is: $x_f$, $y_f$ coordinates of the leftmost eye in the image (the person’s right), $x_s$, $y_s$ coordinates of the rightmost eye, and $x_m$, $y_m$ mouth coordinates. To compute the parameters of the similarity transformation, we can use a least squares solution. For each point $x_i = [ x_i \ y_i ]$ (in this case there are three of them), the point should be mapped to the target location $[ \hat{x}_i \ \hat{y}_i ]$ as

![image.png](attachment:image.png)

More point correspondences would work the same way and only add extra rows to the matrix. The least squares solution is found using linalg.lstsq(). This idea of using least squares solutions is a standard trick that will be used many times in this book. Actually this is the same as used in the DLT algorithm earlier. The code looks like this

In [ ]:
from scipy import linalg

def compute_rigid_transform(refpoints,points):
    """ Computes rotation, scale and translation for
        aligning points to refpoints. """

    A = array([    [points[0], -points[1], 1, 0],
                [points[1],  points[0], 0, 1],
                [points[2], -points[3], 1, 0],
                [points[3],  points[2], 0, 1],
                [points[4], -points[5], 1, 0],
                [points[5],  points[4], 0, 1]])

    y = array([    refpoints[0],
                refpoints[1],
                refpoints[2],
                refpoints[3],
                refpoints[4],
                refpoints[5]])

    # least sq solution to mimimize ||Ax - y||
    a,b,tx,ty = linalg.lstsq(A,y)[0]
    R = array([[a, -b], [b, a]]) # rotation matrix incl scale

    return R,tx,ty

The function returns a rotation matrix with scale as well as translation in the $x$ and $y$ directions. To warp the images and store new aligned, images we can apply ndimage.affine_transform() to each color channel (these are color images). As reference frame, any three point coordinates could be used. Here we will use the landmark
locations in the first image for simplicity:

In [ ]:
from scipy import ndimage
import cv2
import os
import matplotlib

def rigid_alignment(faces,path,plotflag=False):
    """    Align images rigidly and save as new images.
        path determines where the aligned images are saved
        set plotflag=True to plot the images. """

    # take the points in the first image as reference points
    # print(faces.values())
    refpoints = list(faces.values())[0]

    # warp each image using affine transform
    for face in faces:
        points = faces[face]

        R,tx,ty = compute_rigid_transform(refpoints, points)
        T = array([[R[1][1], R[1][0]], [R[0][1], R[0][0]]])

        im = array(Image.open(os.path.join(path,face)))
        im2 = zeros(im.shape, 'uint8')

        # warp each color channel
        for i in range(len(im.shape)):
            im2[:,:,i] = ndimage.affine_transform(im[:,:,i],linalg.inv(T),offset=[-ty,-tx])

        if plotflag:
            cv2.imshow('writeim',im2)
            show()
        #    cv2.waitKey(0)
        #    cv2.destroyAllWindows()

        # crop away border and save aligned images
        h,w = im2.shape[:2]
        border = (w+h)/20

        # crop away border
        # cv2.imwrite(os.path.join(path, 'aligned/'+face),im2[border:h-border,border:w-border,:])
        # cv2.imwrite('Z:\\CompViz\\aligned\\test.jpg',im2)
        cv2.imwrite(os.path.join(path, 'aligned/'+face),im2)

Here we use the cv2.imsave() function to save the aligned images to a sub-directory “aligned”. The following short script will read the XML file containing filenames as keys and points as values and then register all the images to align them with the first one:

In [ ]:
"""
This is the face image registration example from Figure 3-6.
Make sure to create a folder 'aligned' under the jkfaces folder.
"""

# load the location of control points
xml_filename = 'jkfaces.xml'
points = read_points_from_xml(xml_filename)

# register
rigid_alignment(points,'jkfaces/')

If you run this, you should get aligned face images in a sub-directory. Figure 3-6 in Solem shows six sample images before and after registration. The registered images are cropped slightly to remove the undesired black fill pixels that may appear at the borders of the images.

Now let’s see how this affects the mean image. Figure 3-7 shows the mean image for the unaligned face images next to the mean image of the aligned images (note the size difference due to cropping the borders of the aligned images). Although the original images show very little variation in size of the face, rotation and position, the effects on the mean computation are drastic.

Not surprisingly, using badly registered images also has a drastic impact on the computation of principal components. Figure 3-8 in Solem shows the result of PCA on the first 150 images from this set without and with registration. Just as with the mean image, the PCA-modes are blurry. When computing the principal components, we used a mask consisting of an ellipse centered around the mean face position. By multiplying the images with
this mask before stacking them, we can avoid bringing background variations into the PCA-modes. Just replace the line that creates the matrix in the PCA example in Section 1.3 (page 14) of Solem with

immatrix = array([mask*array(Image.open(imlist[i]).convert('L')).flatten()
for i in range(150)],'f')

where mask is a binary image of the same size, already flattened.